# FTS GC pipeline driven by metadata YAML

This notebook reads experiment metadata from `FTS_Robin_metadata.yaml`, builds a chromstream `Experiment` from the `chromatograms/` folder declared in the metadata, integrates peaks using the peak windows in the metadata, converts areas to amounts, applies an internal-standard correction using metadata values, parses reactor log files with `chromstream.parsers.parse_log_file`, and merges results.

Edit `current_experiment` to select a different experiment from the YAML. Make sure the `root` and `folder_name` entries in the metadata point to actual paths on disk.

In [ ]:
# Imports
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import chromstream.parsers as cs_parsers
from chromstream.objects import Experiment
import FTS_Analysis_GC as fts

plt.rcParams['figure.figsize'] = (10, 5)

In [ ]:
# Load metadata and select experiment
metadata_path = Path(r'C:/Users/6855245/OneDrive - Universiteit Utrecht/PhD/Results/FTS catalytic tests/Processed_data/raw_data/FTS_Robin_metadata.yaml')
all_metadata = fts.load_experiment_metadata(str(metadata_path))
# Choose experiment key from YAML
current_experiment = 'RCI_AGL_PVL_006_FTS_Co10TiO2'
if current_experiment not in all_metadata:
    raise KeyError(f"Experiment '{current_experiment}' not found in metadata. Available keys: {list(all_metadata.keys())}")
md = all_metadata[current_experiment]
# Show a compact summary
print('Experiment:', md.get('experiment_name', current_experiment))
print('Root:', md.get('root'))
print('Folder:', md.get('folder_name'))
print('GC subfolder (gc):', md.get('gc'))
print('Log subfolder (log):', md.get('log'))

In [ ]:
# Resolve experiment paths
root_path = Path(md.get('root', '') )
if md.get('folder_name'):
    exp_path = root_path / md['folder_name']
else:
    exp_path = root_path
gc_rel = md.get('gc', 'chromatograms/')
log_rel = md.get('log', 'log/')
chrom_dir = (exp_path / gc_rel).resolve()
log_dir = (exp_path / log_rel).resolve()
print('Experiment folder resolved to:', exp_path)
print('Chromatograms folder:', chrom_dir)
print('Log folder:', log_dir)

In [ ]:
# Build chromstream Experiment from files in chromatograms folder
chrom_files = sorted(chrom_dir.iterdir()) if chrom_dir.exists() else []
print(f'Found {len(chrom_files)} files in {chrom_dir}')
exp = Experiment(name=current_experiment)
for fp in chrom_files:
    try:
        if fp.suffix.lower() == '.txt':
            exp.add_chromatogram(fp)
    except Exception as e:
        print(f'Warning parsing {fp.name}: {e}')

print('Channels:', exp.channel_names)

In [ ]:
# Build peaklist from metadata (map YAML keys to chromstream channel names)
chan_map = {
    'FID_peaks': 'FID',
    'TCDLeft_peaks': 'TCD_AuxLeft',
    'TCDRight_peaks': 'TCD_AuxRight',
    'TCD_peaks': 'TCD_AuxLeft'  # fallback
}
peaklist = {}
two_point_flags = {}
for key, ch_name in chan_map.items():
    if key in md:
        peaks_block = md[key]
        peaks_for_channel = {}
        flags_for_channel = {}
        for pname, pinfo in peaks_block.items():
            # pinfo expected to have 'range' and 'two_point_baseline'
            rng = pinfo.get('range') if isinstance(pinfo, dict) else None
            tp = pinfo.get('two_point_baseline', False) if isinstance(pinfo, dict) else False
            if rng and len(rng) >= 2:
                peaks_for_channel[pname] = (float(rng[0]), float(rng[1]))
                flags_for_channel[pname] = bool(tp)
        if peaks_for_channel:
            peaklist[ch_name] = peaks_for_channel
            two_point_flags[ch_name] = flags_for_channel

print('Constructed peaklist for channels:', list(peaklist.keys()))
print('Example peaks for FID (if present):', peaklist.get('FID'))

In [ ]:
# Integrate using chromstream channel integrators (note: two-point baseline flags are recorded but not applied here)
integrals_list = []
for ch_name, peaks in peaklist.items():
    if ch_name not in exp.channels:
        print(f'Channel {ch_name} not found, skipping')
        continue
    print(f'Integrating {ch_name} with {len(exp.channels[ch_name].chromatograms)} injections')
    df_int = exp.channels[ch_name].integrate_peaks(peaks)
    # ensure Timestamp is datetime
    df_int['Timestamp'] = pd.to_datetime(df_int['Timestamp'])
    integrals_list.append((ch_name, df_int))

# Combine integrals into a single area DataFrame indexed by TOS (minutes)
t0 = min([df['Timestamp'].min() for (_, df) in integrals_list]) if integrals_list else None
area_dfs = []
for ch_name, df in integrals_list:
    df2 = df.copy()
    df2['TOS'] = (pd.to_datetime(df2['Timestamp']) - t0).dt.total_seconds() / 60.0
    df2 = df2.set_index('TOS')
    if 'Timestamp' in df2.columns:
        df2 = df2.drop(columns=['Timestamp'])
    area_dfs.append(df2)
area_total = pd.concat(area_dfs, axis=1).sort_index()
area_total.index.name = 'Time_Point'
print('Area_total shape:', area_total.shape)
area_total.head()

In [ ]:
# Prepare correction factors: try to read from metadata if present; otherwise default to 1.0
# Metadata in the provided YAML does not include explicit correction_factors, so default to 1.0 for all detected peaks
all_peak_names = list(area_total.columns)
correction_factors = {p: 1.0 for p in all_peak_names}
print('Using correction factors (default=1.0) for peaks:', correction_factors)

# Internal standard from metadata if present
internal_standard = md.get('IntStand', md.get('IntStand', 'Ar'))
internal_standard_concentration = md.get('ISConc', md.get('ISConc', None))
print('Internal standard:', internal_standard, 'IS conc:', internal_standard_concentration)

# Convert and IS-correct
amounts = fts.convert_area_to_amount(area_total, correction_factors)
if internal_standard_concentration is not None:
    amounts_corrected = fts.internal_standard_correction(amounts, internal_standard, internal_standard_concentration)
else:
    print('IS concentration not found in metadata; skipping IS correction')
    amounts_corrected = amounts.copy()

amounts_corrected.head()

In [ ]:
# Parse log file(s) using chromstream.parsers.parse_log_file and merge with amounts_corrected
log_files = sorted(log_dir.iterdir()) if log_dir.exists() else []
if not log_files:
    print('No log files found in', log_dir)
    log_df = pd.DataFrame()
else:
    # parse and concatenate multiple log files if present
    log_dfs = []
    for lf in log_files:
        try:
            parsed = cs_parsers.parse_log_file(lf)
            if 'Timestamp' not in parsed.columns and parsed.index.name == 'Timestamp':
                parsed = parsed.reset_index()
            if 'Timestamp' in parsed.columns:
                parsed['Timestamp'] = pd.to_datetime(parsed['Timestamp'])
            log_dfs.append(parsed)
        except Exception as e:
            print(f'Warning parsing log {lf.name}: {e}')
    if log_dfs:
        log_df = pd.concat(log_dfs, ignore_index=True)
        log_df = log_df.sort_values('Timestamp').reset_index(drop=True)
    else:
        log_df = pd.DataFrame()

log_df.head()

In [ ]:
# Merge amounts with log using nearest TOS (fts.parse_logfile_areas expects area_df with TOS index)
if not log_df.empty:
    log_copy = log_df.copy()
    t0 = pd.to_datetime(min(v.injection_time for ch in exp.channels.values() for v in ch.chromatograms.values()))
    log_copy['TOS'] = (pd.to_datetime(log_copy['Timestamp']) - t0).dt.total_seconds() / 60.0
    combined = fts.parse_logfile_areas(amounts_corrected.reset_index(), log_copy)
    combined_df = combined
else:
    combined_df = amounts_corrected.copy()

combined_df.head()

In [ ]:
# Compute conversion for the main reaction gas if present in metadata reactant_conc
reactant = md.get('reaction_gas', 'CO')
reactant_concs = md.get('reactant_conc', {}) or md.get('reactant_conc1', {})
reactant_initial = None
if reactant and reactant_concs and reactant in reactant_concs:
    reactant_initial = float(reactant_concs[reactant])

if reactant_initial is not None and reactant in amounts_corrected.columns:
    combined_df[f'{reactant} conversion (%)'] = fts.calculate_conversion_based_on_reactant(amounts_corrected, reactant, reactant_initial).reindex(combined_df.index)
    print(f'Added conversion for {reactant}')
else:
    print('Could not compute conversion: reactant initial conc or column missing')

combined_df.head()

In [ ]:
# Quick overview plot (try module helper then fallback)
integration_gases = [c for c in amounts_corrected.columns if c not in ['IS_correction_factor']]
gas_flow_columns = [c for c in (log_df.columns if not log_df.empty else []) if 'MFC' in c or 'MFM' in c or 'Flow' in c or 'flow' in c]
try:
    fts.plot_comined_overview(combined_df, gas_flow_columns=gas_flow_columns, experiment_name=current_experiment, integration_gases=integration_gases)
except Exception as e:
    print('Plot helper failed:', e)
    if f'{reactant} conversion (%)' in combined_df.columns:
        combined_df[f'{reactant} conversion (%)'].plot(marker='o')
        plt.xlabel('TOS (min)')
        plt.ylabel(f'{reactant} conversion (%)')
        plt.show()

## Notes and next steps
- This notebook sources peak windows, internal standards and paths from the metadata YAML.
- The `two_point_baseline` flags from metadata are detected but not applied here; if you want two-point baseline subtraction per-peak we can implement a small integration routine that performs the two-point subtraction for those peaks before integrating.
- If you want results exported to Excel, I can add a cell that writes `combined_df.to_excel(...)`.